---
## Scetion 0: Initial setup

**Imports**

In [1]:
# Imports
import sys
from pathlib import Path

# This notebook only orchestrates. Every function and class it calls lives in the
# `macrocircuits` package under src/ -- read or edit it there.
SRC = str(Path.cwd().parent / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import matplotlib.pyplot as plt
from IPython.display import display

# Importing macrocircuits also registers the `swim` and `swim_12_links` tasks with
# the dm_control swimmer suite, which is what lets suite.load() find them below.
from macrocircuits import ensure_tonic

# Clone neuromatch/tonic next to this notebook (once) and put it on the import path.
# It has to run before the tonic-backed imports below (training, models); see
# src/macrocircuits/tonic_setup.py for why tonic isn't a pip install.
ensure_tonic()

# The Swim task rewards swimming forward at _SWIM_SPEED and hides the target the
# stock dm_control swimmer chases. resolve_runs/run_config/run_path turn the runs
# declared in Section 1 into the code strings train() needs and the paths it writes.
from macrocircuits.training import (  # See src/macrocircuits/training.py.
    play_model,
    resolve_runs,
    run_config,
    run_path,
    train,
)
from macrocircuits.plotting import (  # See src/macrocircuits/plotting.py.
    plot_performance,
)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


---
## Scetion 1: Select parameters

**Parameters**

Everything you might want to vary lives here, so the training and plotting cells
below never have to be edited.

`RUNS` declares the training runs to compare — add as many as you like. Each entry
sets only what it varies; `DEFAULTS` fills in the rest:

- `network` — architecture: `'ncap'` or `'mlp'`. The only required key.
- `rl_method` — reinforcement-learning algorithm: `'ppo'`, `'a2c'` or `'trpo'`.
- `n_links` — swimmer body length (`6` → 5 joints, `12` → 11 joints).
- `actor_sizes` / `critic_sizes` — hidden-layer widths of the MLP torsos.
- `action_noise` — exploration std of the NCAP policy head.
- `gradient_clip` — max gradient norm per update (`0` disables).
- `steps` / `save_steps` — training budget and checkpoint interval.
- `label` — the run's directory name and its entry in the plot legend.

`label` defaults to `'<network>_<rl_method>'`, which is what keeps runs apart on
disk. Two runs sharing both — say the same `'mlp'` on `'trpo'` with different
`critic_sizes` — would write into the same directory and overwrite each other's
checkpoints, so `resolve_runs` raises and asks you to label them instead.

In [2]:
# Parameters

# Shared by every run below; each RUNS entry overrides only what it varies.
DEFAULTS = dict(
    # Reinforcement-learning algorithm used to train every network. On-policy tonic
    # agents that fit these stochastic actor-critic models: 'ppo', 'a2c', 'trpo'.
    rl_method='trpo',
    # Swimmer body length (number of rigid links). Registered options:
    #   6  -> 5 joints  (dm_control task 'swim')
    #   12 -> 11 joints (dm_control task 'swim_12_links')
    n_links=6,
    # Hidden-layer sizes of the actor/critic MLP torsos. NCAP's actor is the fixed
    # swimmer circuit, so actor_sizes only affects the MLP baseline; critic_sizes is
    # used by the value network of both.
    actor_sizes=(256, 256),
    critic_sizes=(256, 256),
)

# The runs to train and compare -- as many as you like.
#   'ncap' -- the C. elegans-derived Neural Circuit Architectural Prior (sparse, few weights)
#   'mlp'  -- a generic fully-connected baseline
# NCAP has far fewer parameters and converges sooner, so it needs fewer steps than
# the MLP -- hence the per-run training budget.
RUNS = resolve_runs(
    [
        dict(network='ncap', steps=int(1e5), save_steps=int(5e4)),
        dict(network='mlp', steps=int(5e5), save_steps=int(1e5)),
        # Add more runs here. Anything left out falls back to DEFAULTS, and a run
        # that already differs in network or rl_method needs no label:
        # dict(network='ncap', rl_method='ppo', steps=int(1e5), save_steps=int(5e4)),
        #
        # Runs sharing a network *and* rl_method do need one, since the label is what
        # separates their directories -- and what names them in the plot legend:
        # dict(network='mlp', critic_sizes=(64, 64), label='mlp_small_critic',
        #      steps=int(5e5), save_steps=int(1e5)),
    ],
    defaults=DEFAULTS,
)

# What the training cell will write, so a typo shows up before it costs a run.
for run in RUNS:
    print(f'{run["label"]:<20} {run["steps"]:>9,} steps  ->  {run_path(**run)}')

ncap_trpo              100,000 steps  ->  data/local/experiments/tonic/swimmer-swim/ncap_trpo
mlp_trpo               500,000 steps  ->  data/local/experiments/tonic/swimmer-swim/mlp_trpo


---
## Scetion 2: Training

**Training**

In [3]:
# Training
#
# Train every run declared in RUNS above. run_config() maps each run's parameters
# onto the right model factory, environment and trainer strings -- see
# src/macrocircuits/training.py.
for run in RUNS:
    agent, environment, name, trainer = run_config(**run)
    print(f'\n=== Training {name} ({run["steps"]:,} steps) ===')
    train('import tonic.torch', agent, environment, name=name, trainer=trainer)


=== Training ncap_trpo (100,000 steps) ===


c:\Users\Jakoh\GitHub\Neuromatch_macrocircuits\neuroai-macrocircuits\.venv\Lib\site-packages\gym\spaces\box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


Config file saved to data\local\experiments\tonic\swimmer-swim\ncap_trpo\config.yaml
          Time left:  epoch 0:00:00  total 0:03:24          
actor                                                       
  backtrack steps                                       1.25
  kl                                                 0.00532
  loss                                              -0.00348
critic                                                      
  iterations                                              80
  loss                                                  62.5
  v                                                     32.1
test                                                        
  action                                                    
    max                                                 1.35
    mean                                              0.0296
    min                                                -1.32
    size                                               5,000


c:\Users\Jakoh\GitHub\Neuromatch_macrocircuits\neuroai-macrocircuits\Ressources\tonic\tonic\torch\updaters\optimizers.py:63: RuntimeWarning: invalid value encountered in scalar divide
  p = r + (r_dot_new / r_dot_old) * p


ValueError: Expected parameter loc (Tensor of shape (4096, 5)) of distribution Normal(loc: torch.Size([4096, 5]), scale: torch.Size([4096, 5])) to satisfy the constraint Real(), but found invalid values:
tensor([[nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        ...,
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan],
        [nan, nan, nan, nan, nan]], grad_fn=<CatBackward0>)

---
## Scetion 3: Visualization

**Visualization of the rewards and the behaviour**

In [ ]:
# Visualization

In [ ]:
# rerun this cell if it displays more than the graphs
%matplotlib inline
fig, ax = plt.subplots()

# Learning curve of every run trained above, from the same RUNS list, so the paths
# always match what the training cell wrote. plot_performance names each curve after
# the last folder of its path -- i.e. the run's label.
paths = [run_path(**run) for run in RUNS]

# The legend names every curve, so only spell the runs out in the title while they
# still fit on one line.
if len(RUNS) <= 3:
    title = ' v/s '.join(run['label'].upper() for run in RUNS)
else:
    title = f'{len(RUNS)} runs'
plot_performance(paths, ax=ax, title=title)
plt.tight_layout()
plt.show()

In [ ]:
# Replay the last checkpoint of every run and render it as video. To watch a single
# run instead, call play_model on just its path, e.g. play_model(run_path(**RUNS[0])).
for run in RUNS:
    print(f'\n=== {run["label"]} ===')
    display(play_model(run_path(**run)))